In [1]:
#cell 1

!pip -q install -U "transformers>=4.51.0" "sentence-transformers>=2.7.0" "chromadb>=1.0.0" "tqdm" "numpy" "pandas"

In [2]:
#cell 2

import os
import json
import time
import gc
import random
import shutil
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import chromadb
from sentence_transformers import SentenceTransformer

In [3]:
#cell 3

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
#cell 4

PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
RAG_DIR = PROJECT_DIR / "RAG"
EVIDENCE_DIR = RAG_DIR / "evidence"

MODEL_NAME = "Qwen/Qwen3-Embedding-8B"
EXPECTED_EMBEDDING_DIM = 4096

# This must be exactly 6 based on your requested baseline RAG setup.
RETRIEVAL_TOP_K = 6

# The shuffle is done separately for each dataset.
SHUFFLE_SEED = 42

# Conservative batch size, similar to your previous notebook.
QUERY_ENCODE_BATCH_SIZE = 16

EXPECTED_NUM_QUESTIONS_PER_DATASET = 1000

DATASETS = {
    "hotpotqa": {
        "question_json_path": PROJECT_DIR / "hotpotqa_dev_2017wiki_1000_converted.json",
        "db_dir": RAG_DIR / "hotpotqa",
        "collection_name": "chunks",
        "output_json_path": EVIDENCE_DIR / "hotpotqa_evidence.json",
    },
    "2wikimultihopqa": {
        "question_json_path": PROJECT_DIR / "2wikimultihopqa_dev_2020wiki_1000_converted.json",
        "db_dir": RAG_DIR / "2wikimultihopqa",
        "collection_name": "chunks",
        "output_json_path": EVIDENCE_DIR / "2wikimultihopqa_evidence.json",
    },
}

print("Project directory:", PROJECT_DIR)
print("RAG directory:", RAG_DIR)
print("Evidence output directory:", EVIDENCE_DIR)

for dataset_name, cfg in DATASETS.items():
    print("=" * 80)
    print("Dataset:", dataset_name)
    print("Question JSON:", cfg["question_json_path"])
    print("Vector DB:", cfg["db_dir"])
    print("Collection name:", cfg["collection_name"])
    print("Output JSON:", cfg["output_json_path"])

Project directory: /content/drive/MyDrive/final_project
RAG directory: /content/drive/MyDrive/final_project/RAG
Evidence output directory: /content/drive/MyDrive/final_project/RAG/evidence
Dataset: hotpotqa
Question JSON: /content/drive/MyDrive/final_project/hotpotqa_dev_2017wiki_1000_converted.json
Vector DB: /content/drive/MyDrive/final_project/RAG/hotpotqa
Collection name: chunks
Output JSON: /content/drive/MyDrive/final_project/RAG/evidence/hotpotqa_evidence.json
Dataset: 2wikimultihopqa
Question JSON: /content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_converted.json
Vector DB: /content/drive/MyDrive/final_project/RAG/2wikimultihopqa
Collection name: chunks
Output JSON: /content/drive/MyDrive/final_project/RAG/evidence/2wikimultihopqa_evidence.json


In [5]:
#cell 5

assert PROJECT_DIR.exists(), f"PROJECT_DIR does not exist: {PROJECT_DIR}"
assert RAG_DIR.exists(), f"RAG_DIR does not exist: {RAG_DIR}"

EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
print("Ready evidence directory:", EVIDENCE_DIR)

for dataset_name, cfg in DATASETS.items():
    assert cfg["question_json_path"].exists(), (
        f"Missing input question JSON for {dataset_name}: {cfg['question_json_path']}"
    )

    assert cfg["db_dir"].exists(), (
        f"Missing vector database directory for {dataset_name}: {cfg['db_dir']}"
    )

    chroma_sqlite_path = cfg["db_dir"] / "chroma.sqlite3"
    assert chroma_sqlite_path.exists(), (
        f"Missing Chroma sqlite file for {dataset_name}: {chroma_sqlite_path}"
    )

    manifest_path = cfg["db_dir"] / "build_manifest.json"
    assert manifest_path.exists(), (
        f"Missing build manifest for {dataset_name}: {manifest_path}"
    )

    print(f"{dataset_name}: paths are valid.")

Ready evidence directory: /content/drive/MyDrive/final_project/RAG/evidence
hotpotqa: paths are valid.
2wikimultihopqa: paths are valid.


In [6]:
#cell 6

def load_json_list(json_path: Path, dataset_name: str) -> List[Dict[str, Any]]:
    """Load a JSON file whose root must be a list of records."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{dataset_name}: JSON root must be a list.")

    if len(data) != EXPECTED_NUM_QUESTIONS_PER_DATASET:
        raise ValueError(
            f"{dataset_name}: expected {EXPECTED_NUM_QUESTIONS_PER_DATASET} records, "
            f"but found {len(data)}."
        )

    return data


def validate_question_record(record: Dict[str, Any], dataset_name: str, index: int) -> None:
    """Validate that one question record has all required keys."""
    required_keys = {"type", "question", "answer", "supports"}

    if not isinstance(record, dict):
        raise ValueError(f"{dataset_name}: record {index} is not a dictionary.")

    missing_keys = required_keys - set(record.keys())
    if missing_keys:
        raise ValueError(
            f"{dataset_name}: record {index} is missing required keys: {missing_keys}"
        )

    if not isinstance(record["question"], str) or not record["question"].strip():
        raise ValueError(f"{dataset_name}: record {index} has an invalid question.")


def keep_required_fields(record: Dict[str, Any]) -> Dict[str, Any]:
    """Keep only the required original fields before adding evidence_chunk."""
    return {
        "type": record["type"],
        "question": record["question"],
        "answer": record["answer"],
        "supports": record["supports"],
    }


def load_validate_and_shuffle_questions(
    dataset_name: str,
    json_path: Path,
    seed: int
) -> List[Dict[str, Any]]:
    """Load, validate, keep required fields, and shuffle records for one dataset."""
    records = load_json_list(json_path, dataset_name)

    for i, record in enumerate(records):
        validate_question_record(record, dataset_name, i)

    selected_records = [keep_required_fields(record) for record in records]

    # Shuffle is intentionally done independently for each dataset.
    rng = random.Random(seed)
    rng.shuffle(selected_records)

    print(f"{dataset_name}: loaded, validated, selected fields, and shuffled {len(selected_records):,} records.")

    return selected_records

In [7]:
#cell 7

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = SentenceTransformer(MODEL_NAME, device=device)

probe_text = ["Title: Test\nText: This is a short test document."]
probe_embedding = model.encode(
    probe_text,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("Probe embedding shape:", probe_embedding.shape)

assert probe_embedding.shape[1] == EXPECTED_EMBEDDING_DIM, (
    f"Expected embedding dimension {EXPECTED_EMBEDDING_DIM}, "
    f"but got {probe_embedding.shape[1]}"
)

del probe_embedding
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Probe embedding shape: (1, 4096)


In [8]:
#cell 8

def load_manifest(dataset_name: str, db_dir: Path) -> Dict[str, Any]:
    """Load and validate the vector database build manifest."""
    manifest_path = db_dir / "build_manifest.json"

    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    if manifest.get("dataset") != dataset_name:
        raise ValueError(
            f"Manifest dataset mismatch. Expected {dataset_name}, "
            f"but found {manifest.get('dataset')} in {manifest_path}"
        )

    if manifest.get("model_name") != MODEL_NAME:
        raise ValueError(
            f"{dataset_name}: model mismatch. Expected {MODEL_NAME}, "
            f"but manifest has {manifest.get('model_name')}"
        )

    if int(manifest.get("embedding_dimension", -1)) != EXPECTED_EMBEDDING_DIM:
        raise ValueError(
            f"{dataset_name}: embedding dimension mismatch. "
            f"Expected {EXPECTED_EMBEDDING_DIM}, "
            f"but manifest has {manifest.get('embedding_dimension')}"
        )

    if manifest.get("similarity_space") != "cosine":
        raise ValueError(
            f"{dataset_name}: expected cosine similarity space, "
            f"but manifest has {manifest.get('similarity_space')}"
        )

    return manifest


def load_collection(dataset_name: str, cfg: Dict[str, Any]):
    """Load the Chroma collection for exactly one dataset and validate it."""
    manifest = load_manifest(dataset_name, cfg["db_dir"])

    client = chromadb.PersistentClient(path=str(cfg["db_dir"]))
    collection = client.get_collection(name=cfg["collection_name"])

    collection_count = collection.count()

    if collection_count != int(manifest["collection_count"]):
        raise ValueError(
            f"{dataset_name}: collection count mismatch. "
            f"Manifest says {manifest['collection_count']}, "
            f"but Chroma collection has {collection_count}."
        )

    sample = collection.get(
        limit=1,
        include=["metadatas"]
    )

    if not sample["ids"]:
        raise ValueError(f"{dataset_name}: collection is empty.")

    sample_metadata = sample["metadatas"][0]
    sample_dataset = sample_metadata.get("dataset")

    if sample_dataset != dataset_name:
        raise ValueError(
            f"{dataset_name}: sample metadata dataset mismatch. "
            f"Expected {dataset_name}, but found {sample_dataset}."
        )

    print(f"{dataset_name}: collection loaded successfully.")
    print(f"{dataset_name}: collection count = {collection_count:,}")

    return collection

In [9]:
#cell 9

def extract_text_from_chroma_document(document: str) -> str:
    """Extract only the Text part from the stored Chroma document."""
    if not isinstance(document, str):
        return ""

    marker = "\nText:"
    if marker in document:
        return document.split(marker, 1)[1].strip()

    if document.startswith("Text:"):
        return document[len("Text:"):].strip()

    return document.strip()


def extract_title_from_chroma_document(document: str) -> str:
    """Extract the Title part from the stored Chroma document as a fallback."""
    if not isinstance(document, str):
        return ""

    if document.startswith("Title:") and "\nText:" in document:
        title_part = document.split("\nText:", 1)[0]
        return title_part[len("Title:"):].strip()

    return ""


def make_evidence_chunk(document: str, metadata: Dict[str, Any]) -> Dict[str, str]:
    """Build one evidence chunk with only title and text."""
    title = ""

    if isinstance(metadata, dict):
        title = str(metadata.get("title", "")).strip()

    if not title:
        title = extract_title_from_chroma_document(document)

    text = extract_text_from_chroma_document(document)

    return {
        "title": title,
        "text": text,
    }

In [10]:
#cell 10

def retrieve_evidence_chunks_for_questions(
    dataset_name: str,
    collection,
    questions: List[str],
    top_k: int = RETRIEVAL_TOP_K,
    batch_size: int = QUERY_ENCODE_BATCH_SIZE
) -> List[List[Dict[str, str]]]:
    """Retrieve top-k evidence chunks for each question from one dataset collection."""
    all_evidence_chunks = []

    for start_idx in tqdm(
        range(0, len(questions), batch_size),
        desc=f"Embedding + retrieving {dataset_name}"
    ):
        batch_questions = questions[start_idx:start_idx + batch_size]

        query_embeddings = model.encode(
            batch_questions,
            batch_size=len(batch_questions),
            prompt_name="query",
            convert_to_numpy=True,
            show_progress_bar=False
        )

        if query_embeddings.shape[1] != EXPECTED_EMBEDDING_DIM:
            raise ValueError(
                f"{dataset_name}: expected query embedding dim {EXPECTED_EMBEDDING_DIM}, "
                f"but got {query_embeddings.shape[1]}"
            )

        query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

        results = collection.query(
            query_embeddings=query_embeddings.tolist(),
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )

        batch_ids = results["ids"]
        batch_documents = results["documents"]
        batch_metadatas = results["metadatas"]

        if len(batch_ids) != len(batch_questions):
            raise ValueError(
                f"{dataset_name}: Chroma returned {len(batch_ids)} result groups "
                f"for {len(batch_questions)} questions."
            )

        for local_i in range(len(batch_questions)):
            ids_for_question = batch_ids[local_i]
            docs_for_question = batch_documents[local_i]
            metas_for_question = batch_metadatas[local_i]

            if len(ids_for_question) != top_k:
                raise ValueError(
                    f"{dataset_name}: expected {top_k} retrieved chunks, "
                    f"but got {len(ids_for_question)}."
                )

            evidence_chunks = []

            for chunk_id, document, metadata in zip(
                ids_for_question,
                docs_for_question,
                metas_for_question
            ):
                metadata_dataset = metadata.get("dataset") if isinstance(metadata, dict) else None

                if metadata_dataset != dataset_name:
                    raise ValueError(
                        f"Dataset contamination detected. "
                        f"Current dataset is {dataset_name}, "
                        f"but retrieved chunk {chunk_id} has dataset metadata {metadata_dataset}."
                    )

                evidence_chunks.append(
                    make_evidence_chunk(document=document, metadata=metadata)
                )

            all_evidence_chunks.append(evidence_chunks)

        del query_embeddings
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if len(all_evidence_chunks) != len(questions):
        raise ValueError(
            f"{dataset_name}: final evidence count mismatch. "
            f"Expected {len(questions)}, got {len(all_evidence_chunks)}."
        )

    return all_evidence_chunks

In [11]:
#cell 11

def build_evidence_file_for_dataset(
    dataset_name: str,
    cfg: Dict[str, Any],
    seed: int
) -> Dict[str, Any]:
    """Build and save the evidence JSON file for one dataset."""
    start_time = time.time()

    print("=" * 80)
    print(f"Starting dataset: {dataset_name}")

    records = load_validate_and_shuffle_questions(
        dataset_name=dataset_name,
        json_path=cfg["question_json_path"],
        seed=seed
    )

    collection = load_collection(dataset_name, cfg)

    questions = [record["question"] for record in records]

    evidence_chunks_per_question = retrieve_evidence_chunks_for_questions(
        dataset_name=dataset_name,
        collection=collection,
        questions=questions,
        top_k=RETRIEVAL_TOP_K,
        batch_size=QUERY_ENCODE_BATCH_SIZE
    )

    output_records = []

    for record, evidence_chunks in zip(records, evidence_chunks_per_question):
        output_record = {
            "type": record["type"],
            "question": record["question"],
            "answer": record["answer"],
            "supports": record["supports"],
            "evidence_chunk": evidence_chunks,
        }

        output_records.append(output_record)

    cfg["output_json_path"].parent.mkdir(parents=True, exist_ok=True)

    with open(cfg["output_json_path"], "w", encoding="utf-8") as f:
        json.dump(output_records, f, ensure_ascii=False, indent=2)

    elapsed_seconds = time.time() - start_time

    print(f"{dataset_name}: saved evidence file to {cfg['output_json_path']}")
    print(f"{dataset_name}: number of records = {len(output_records):,}")
    print(f"{dataset_name}: elapsed seconds = {elapsed_seconds:.2f}")

    return {
        "dataset": dataset_name,
        "input_question_json": str(cfg["question_json_path"]),
        "vector_db_dir": str(cfg["db_dir"]),
        "collection_name": cfg["collection_name"],
        "output_json": str(cfg["output_json_path"]),
        "num_records": len(output_records),
        "top_k": RETRIEVAL_TOP_K,
        "shuffle_seed": seed,
        "elapsed_seconds": elapsed_seconds,
    }

In [12]:
#cell 12

build_summaries = []

# The two datasets are processed separately.
# Each dataset uses only its own JSON file and only its own Chroma vector database.
for dataset_name in ["hotpotqa", "2wikimultihopqa"]:
    summary = build_evidence_file_for_dataset(
        dataset_name=dataset_name,
        cfg=DATASETS[dataset_name],
        seed=SHUFFLE_SEED
    )
    build_summaries.append(summary)

summary_df = pd.DataFrame(build_summaries)
display(summary_df)

Starting dataset: hotpotqa
hotpotqa: loaded, validated, selected fields, and shuffled 1,000 records.
hotpotqa: collection loaded successfully.
hotpotqa: collection count = 35,029


Embedding + retrieving hotpotqa:   0%|          | 0/63 [00:00<?, ?it/s]

hotpotqa: saved evidence file to /content/drive/MyDrive/final_project/RAG/evidence/hotpotqa_evidence.json
hotpotqa: number of records = 1,000
hotpotqa: elapsed seconds = 90.15
Starting dataset: 2wikimultihopqa
2wikimultihopqa: loaded, validated, selected fields, and shuffled 1,000 records.
2wikimultihopqa: collection loaded successfully.
2wikimultihopqa: collection count = 12,685


Embedding + retrieving 2wikimultihopqa:   0%|          | 0/63 [00:00<?, ?it/s]

2wikimultihopqa: saved evidence file to /content/drive/MyDrive/final_project/RAG/evidence/2wikimultihopqa_evidence.json
2wikimultihopqa: number of records = 1,000
2wikimultihopqa: elapsed seconds = 58.40


,dataset,input_question_json,vector_db_dir,collection_name,output_json,num_records,top_k,shuffle_seed,elapsed_seconds
0,hotpotqa,/content/drive/MyDrive/final_project/hotpotqa_...,/content/drive/MyDrive/final_project/RAG/hotpotqa,chunks,/content/drive/MyDrive/final_project/RAG/evide...,1000,6,42,90.148860
1,2wikimultihopqa,/content/drive/MyDrive/final_project/2wikimult...,/content/drive/MyDrive/final_project/RAG/2wiki...,chunks,/content/drive/MyDrive/final_project/RAG/evide...,1000,6,42,58.399959


In [13]:
#cell 13

def verify_evidence_file(dataset_name: str, output_json_path: Path) -> None:
    """Verify the saved evidence JSON file."""
    with open(output_json_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    if not isinstance(records, list):
        raise ValueError(f"{dataset_name}: output JSON root must be a list.")

    if len(records) != EXPECTED_NUM_QUESTIONS_PER_DATASET:
        raise ValueError(
            f"{dataset_name}: expected {EXPECTED_NUM_QUESTIONS_PER_DATASET} output records, "
            f"but found {len(records)}."
        )

    required_keys = {"type", "question", "answer", "supports", "evidence_chunk"}

    for i, record in enumerate(records):
        missing_keys = required_keys - set(record.keys())
        if missing_keys:
            raise ValueError(
                f"{dataset_name}: output record {i} is missing keys: {missing_keys}"
            )

        evidence_chunk = record["evidence_chunk"]

        if not isinstance(evidence_chunk, list):
            raise ValueError(f"{dataset_name}: evidence_chunk in record {i} must be a list.")

        if len(evidence_chunk) != RETRIEVAL_TOP_K:
            raise ValueError(
                f"{dataset_name}: record {i} has {len(evidence_chunk)} evidence chunks, "
                f"but expected {RETRIEVAL_TOP_K}."
            )

        for j, chunk in enumerate(evidence_chunk):
            if set(chunk.keys()) != {"title", "text"}:
                raise ValueError(
                    f"{dataset_name}: evidence chunk {j} in record {i} must contain only "
                    f"'title' and 'text'. Found keys: {set(chunk.keys())}"
                )

    print("=" * 80)
    print(f"{dataset_name}: verification passed.")
    print("Output path:", output_json_path)
    print("Number of records:", len(records))
    print("First record keys:", list(records[0].keys()))
    print("First question:", records[0]["question"])
    print("First evidence titles:")
    for rank, chunk in enumerate(records[0]["evidence_chunk"], start=1):
        print(f"{rank}. {chunk['title']}")


for dataset_name, cfg in DATASETS.items():
    verify_evidence_file(
        dataset_name=dataset_name,
        output_json_path=cfg["output_json_path"]
    )

hotpotqa: verification passed.
Output path: /content/drive/MyDrive/final_project/RAG/evidence/hotpotqa_evidence.json
Number of records: 1000
First record keys: ['type', 'question', 'answer', 'supports', 'evidence_chunk']
First question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
First evidence titles:
1. Bedknobs and Broomsticks
2. The Muppet Christmas Carol
3. Bedknobs and Broomsticks
4. The Muppet Christmas Carol
5. Bedknobs and Broomsticks
6. The Muppet Christmas Carol
2wikimultihopqa: verification passed.
Output path: /content/drive/MyDrive/final_project/RAG/evidence/2wikimultihopqa_evidence.json
Number of records: 1000
First record keys: ['type', 'question', 'answer', 'supports', 'evidence_chunk']
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
First evidence titles:
1. John Jaffer Janardhanan
2. A. Jagannathan
3. Kamalakara Kameswara Rao
4. Kamakalawa
5. J. Sasikumar
6. Kamalakara K

In [14]:
#cell 14

print("Evidence folder contents:")

for item in sorted(EVIDENCE_DIR.iterdir()):
    print(" -", item.name)

Evidence folder contents:
 - 2wikimultihopqa_evidence.json
 - hotpotqa_evidence.json
